# Phase 4: Local Reranking for RAG

Goal: improve retrieval quality by adding a reranking step after initial vector search.

In earlier phases, we built local RAG, tuned chunking, and evaluated configs. Now we keep the same local stack but add a second retrieval-quality layer.


## Phase 4 Context

Basic vector retrieval is fast, but it can return chunks that are semantically close without being the best evidence for the question.

Reranking changes the retrieval flow:

```text
question
→ retrieve many candidate chunks with FAISS
→ reranker scores each candidate more carefully
→ keep the top few reranked chunks
→ send those chunks to the LLM
```

Short version:

```text
FAISS = fast broad candidate retrieval
Reranker = slower but more precise final ordering
```

## Agenda

1. Review the original `reranking.ipynb` conceptually.
2. Rebuild our local baseline RAG pipeline.
3. Retrieve more candidates than we plan to send to the LLM.
4. Load a local Hugging Face cross-encoder reranker.
5. Score `(question, chunk)` pairs with the reranker.
6. Compare initial FAISS order vs reranked order.
7. Generate answers using reranked context.
8. Evaluate baseline retrieval vs reranked retrieval using our local eval loop.


## Why This Is Production-Relevant

Production RAG often separates retrieval into two stages:

```text
candidate retrieval
→ get enough potentially useful chunks quickly

reranking
→ reorder candidates by deeper relevance
```

This improves precision: the LLM sees better evidence, which can improve correctness and faithfulness.

Tradeoff: reranking adds latency, so production systems measure whether quality improves enough to justify the extra time.


## Reference Notebook

Use this only as a concept reference:

```text
all_rag_techniques/reranking.ipynb
```

The original notebook includes OpenAI and Colab setup. This local version uses:

```text
Hugging Face embeddings
FAISS
Hugging Face CrossEncoder reranker
Ollama/Gemma for generation and judging
```

In [1]:
from pathlib import Path
import json
import re
import time

import ollama
from sentence_transformers import CrossEncoder

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

MODEL_NAME = "gemma4:e4b"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"


/Users/jatin.sangwan/dev/RAG_Techniques/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jatin.sangwan/dev/RAG_Techniques/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# adding a baseline RAG
  
pdf_path = Path("../data/Understanding_Climate_Change.pdf")
loader = PyPDFLoader(pdf_path)

documents = loader.load()

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap = 200,
        length_function = len
)

chunks = text_splitter.split_documents(documents) # chunks made
vector_store = FAISS.from_documents(chunks, embedding_model) 

print("PDF exists:", pdf_path.exists())
print("Page documents:", len(documents))
print("Chunks:", len(chunks))
print("Vector store:", type(vector_store))

PDF exists: True
Page documents: 33
Chunks: 97
Vector store: <class 'langchain_community.vectorstores.faiss.FAISS'>


In [7]:
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Reranker model:", RERANKER_MODEL_NAME)

Reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


In [13]:
print(type(reranker))
print([ele for ele in dir(reranker) if not ele.startswith("__")])
print(help(reranker))

<class 'sentence_transformers.cross_encoder.CrossEncoder.CrossEncoder'>
['T_destination', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_create_model_card', '_create_repo', '_eval_during_training', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_files_timestamps', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_model', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_model_card_text', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_pre_hook', '_register_state_dict_hook', '_replicate_for_data_parallel', '_save_to_state_dict', '_slow_forward', '_state_dict_hooks', '_state_dict_pre_hooks', '_target_device', '_upload_modified_files', '_version', '_wrapped_call_impl'

In [25]:
query = "What are the impacts of climate change on biodiversity"

candidate_docs = vector_store.similarity_search(query, k=10)

# now we need to make pairs for the re-ranking
pairs = [[query, ele.page_content] for ele in candidate_docs]

print(pairs[0])
print(len(pairs))
print(len(pairs[0]))

scores = reranker.predict(pairs)
print(scores[0])
print(len(scores))

for i,score in enumerate(scores, start=1):
    print(f"this is: {i} with score: {score}")

['What are the impacts of climate change on biodiversity', 'Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species \ndistributions, and impacting ecosystem functions. Forests, grasslands, and deserts are \nexperiencing shifts in plant and animal species composition. These changes can lead to a loss \nof biodiversity and disrupt ecological balance. \nMarine Ecosystems \nMarine ecosystems are highly vulnerable to climate change. Rising sea temperatures, ocean \nacidification, and changing currents affect marine biodiversity, from coral reefs to deep-sea \nhabitats. Species migration and changes in reproductive cycles can disrupt marine food webs \nand fisheries.']
10
2
8.230602
10
this is: 1 with score: 8.230602264404297
this is: 2 with score: 6.220482349395752
this is: 3 with score: 3.6761975288391113
this is: 4 with score: 2.019359827041626
this is: 5 with score: 5.068264961242676
this is: 6 with score: 5.722705841064453
this is: 7 with score: 1.

In [ ]:
scored_docs = list(zip(candidate_docs, scores))
reranked_docs = sorted(scored_docs, key=lambda item: item[1], reverse=True)

print("Initial FAISS top 3:")
for i, doc in enumerate(candidate_docs[:3], start=1):
    print(f"\n--- FAISS {i} ---")
    print(doc.page_content[:300])

print("\nReranked top 3:")
for i, (doc, score) in enumerate(reranked_docs[:3], start=1):
    print(f"\n--- Reranked {i} | score={score:.4f} ---")
    print(doc.page_content[:300])

In [42]:
temp = zip(candidate_docs, scores)
print(temp) 
print(type(temp))
# zip returns a zip object, and that zip object is an iterator.
# print(list(temp))

zipped_list = list(temp)
print(zipped_list)
# doc, score = zipped_list[0]

# print(type(doc))
# print(type(score))

# print(score)
# print(doc.metadata)
# print(doc.page_content[:300])

reranked_list = sorted(zipped_list, key=lambda item: item[1], reverse=True)

print(reranked_list)



<class 'zip'>
[(Document(id='5e9c2c4f-350e-417d-9900-7aa55d382d31', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '../data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 12, 'page_label': '13'}, page_content='Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species \ndistributions, and impacting ecosystem functions. Forests, grasslands, and deserts are \nexperiencing shifts in plant and animal species composition. These changes can lead to a loss \nof biodiversity and disrupt ecological balance. \nMarine Ecosystems \nMarine ecosystems are highly vulnerable to climate change. Rising sea temperatures, ocean \nacidification, and changing currents affect marine biodiversity, from coral reefs to deep-sea \nhabitats. Species migration and changes in reproductive cycles can disrupt marine food we

In [43]:
def retrieve_then_rerank(query, vector_store, candidate_k=10, final_k=3):
    candidate_docs = vector_store.similarity_search(query, k=candidate_k)

    pairs = [
        [query, doc.page_content]
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(candidate_docs, scores))

    reranked_docs = sorted(
        scored_docs,
        key=lambda item: item[1],
        reverse=True,
    )

    return reranked_docs[:final_k]

In [44]:
query = "What are the impacts of climate change on biodiversity?"

top_reranked = retrieve_then_rerank(
    query=query,
    vector_store=vector_store,
    candidate_k=10,
    final_k=3,
)

for i, (doc, score) in enumerate(top_reranked, start=1):
    print(f"\n--- Reranked {i} | score={score:.4f} ---")
    print(doc.page_content[:500])


--- Reranked 1 | score=7.5049 ---
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shifts in plant and animal species composition. These changes can lead to a loss 
of biodiversity and disrupt ecological balance. 
Marine Ecosystems 
Marine ecosystems are highly vulnerable to climate change. Rising sea temperatures, ocean 
acidification, and changing currents affect marine biodiversit

--- Reranked 2 | score=5.7396 ---
protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation strategies must account for climate change impacts to be effective. This 
includes identifying climate refugia, areas less affected by climate change, and prioritizing 
them for protection. Adaptive management practices ensure that conservation efforts remain 
effective under changing conditions. 
Biodiversity in Climate Policy 
Incorporating biod

In [52]:
def run_rag_with_reranking(question, vector_store, candidate_k=10, final_k=3):
    reranked_docs_with_scores = retrieve_then_rerank(
        query=question,
        vector_store=vector_store,
        candidate_k=candidate_k,
        final_k=final_k,
    )

    reranked_docs = [
        doc
        for doc, score in reranked_docs_with_scores
    ]

    context = "\n\n".join(
        doc.page_content
        for doc in reranked_docs
    )

    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say: "I don't know based on the provided context."

Question:
{question}

Context:
{context}

Answer in one short sentence:
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return {
        "question": question,
        "generated_answer": response.message.content,
        "context": context,
        "reranked_docs_with_scores": reranked_docs_with_scores,
        "reranked_docs": reranked_docs,
    }

In [55]:
result = run_rag_with_reranking(
    question="What are the impacts of climate change on biodiversity?",
    vector_store=vector_store,
    candidate_k=10,
    final_k=3,
)

print("Question:")
print(result["question"])

print("\nGenerated answer:")
print(result["generated_answer"])

print("\nReranker scores:")
for i, (doc, score) in enumerate(result["reranked_docs_with_scores"], start=1):
    print(f"{i}. {score:.4f}")

Question:
What are the impacts of climate change on biodiversity?

Generated answer:
Climate change shifts habitat ranges and changes species distributions in terrestrial ecosystems, causing potential losses of biodiversity and disruptions to ecological balance, while also impacting marine biodiversity through factors like rising sea temperatures and ocean acidification.

Reranker scores:
1. 7.5049
2. 5.7396
3. 5.5052


In [56]:
def run_baseline_rag(question, vector_store, k=3):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say: "I don't know based on the provided context."

Question:
{question}

Context:
{context}

Answer in one short sentence:
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return {
        "question": question,
        "generated_answer": response.message.content,
        "context": context,
        "retrieved_docs": retrieved_docs,
    }

In [57]:
question = "What are the impacts of climate change on biodiversity?"

baseline_result = run_baseline_rag(
    question=question,
    vector_store=vector_store,
    k=3,
)

reranked_result = run_rag_with_reranking(
    question=question,
    vector_store=vector_store,
    candidate_k=10,
    final_k=3,
)

print("Question:")
print(question)

print("\nBaseline answer:")
print(baseline_result["generated_answer"])

print("\nReranked answer:")
print(reranked_result["generated_answer"])

Question:
What are the impacts of climate change on biodiversity?

Baseline answer:
Climate change alters ecosystems by shifting habitat ranges and changing species distributions, leading to a loss of biodiversity and disrupting ecological balance in both terrestrial and marine habitats.

Reranked answer:
Climate change alters terrestrial ecosystems by shifting habitat ranges and causing a loss of biodiversity, while rising sea temperatures and ocean acidification affect marine life, disrupting food webs and fisheries.


FAISS retrieves candidate chunks
CrossEncoder scores query-chunk pairs
sorted(...) reorders by score
run_rag_with_reranking(...) generates using reranked chunks

In [58]:
def show_reranking_comparison(query, vector_store, candidate_k=10, final_k=3):
    candidate_docs = vector_store.similarity_search(query, k=candidate_k)

    pairs = [
        [query, doc.page_content]
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(candidate_docs, scores))

    reranked_docs = sorted(
        scored_docs,
        key=lambda item: item[1],
        reverse=True,
    )

    print("=" * 80)
    print("Query:")
    print(query)

    print("\nInitial FAISS order:")
    for i, doc in enumerate(candidate_docs[:final_k], start=1):
        print(f"\n--- FAISS {i} ---")
        print("Page:", doc.metadata.get("page_label") or doc.metadata.get("page"))
        print(doc.page_content[:350])

    print("\nReranked order:")
    for i, (doc, score) in enumerate(reranked_docs[:final_k], start=1):
        print(f"\n--- Reranked {i} | score={score:.4f} ---")
        print("Page:", doc.metadata.get("page_label") or doc.metadata.get("page"))
        print(doc.page_content[:350])

    return {
        "candidate_docs": candidate_docs,
        "scored_docs": scored_docs,
        "reranked_docs": reranked_docs,
    }

candidate_k
final_k
reranker model -  adds latency

In [59]:
query = "What are the impacts of climate change on biodiversity?"

for candidate_k in [5, 10, 20]:
    print("\n" + "=" * 80)
    print("candidate_k:", candidate_k)

    result = show_reranking_comparison(
        query=query,
        vector_store=vector_store,
        candidate_k=candidate_k,
        final_k=3,
    )


candidate_k: 5
Query:
What are the impacts of climate change on biodiversity?

Initial FAISS order:

--- FAISS 1 ---
Page: 13
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shifts in plant and animal species composition. These changes can lead to a loss 
of biodiversity and disrupt ecological balance. 
Marine Ecosystems 
Mar

--- FAISS 2 ---
Page: 14
protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation strategies must account for climate change impacts to be effective. This 
includes identifying climate refugia, areas less affected by climate change, and prioritizing 
them for protection. Adaptive management practices ensure that conservation efforts remain 
effective

--- FAISS 3 ---
Page: 11
The economic costs of climate change include damage to infrastructure, reduced agricultural 
productivity, health care 